In [1]:
###############
# Directories #
###############
directory = r'C:\Users\20234503\Dropbox\PhD Project\Research\Weather and Strikes\Project' 
##########
#Packages#
##########
import re
import json
import pandas as pd

import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from collections import Counter
import string

import geopandas as gpd
from shapely.ops import unary_union

from shapely.geometry import Point
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
from matplotlib.patches import Patch

# --- keep this near your imports ---
import types
import inspect
import sys

def clear_env(keep=None):
    """
    Delete user-defined globals while keeping:
      - all imported modules (e.g. pd, np, gpd, plt, string, etc.)
      - functions/classes imported via `from ... import ...` (e.g. unary_union, Counter, Patch)
      - anything explicitly listed in `keep`
      - underscore-prefixed names and this function itself
    """
    if keep is None:
        keep = []
    keep = set(keep) | {"clear_env"}

    g = globals()
    for name, val in list(g.items()):
        # never touch dunders, explicit keepers, or this function
        if name.startswith("_") or name in keep:
            continue

        # keep modules (e.g., pd, np, gpd, plt, string, re, json, sys, types, etc.)
        if isinstance(val, types.ModuleType):
            continue

        # keep imported functions/classes (e.g., unary_union, Counter, Patch, product)
        # (callables or types) whose defining module is not __main__
        if (callable(val) or isinstance(val, type)):
            mod = inspect.getmodule(val)
            if mod is not None and mod.__name__ != "__main__":
                continue

        # otherwise, remove it (likely user-created data/vars/objects)
        try:
            del g[name]
        except Exception:
            pass

# Checked if code is correct 13/11/2025 

In [3]:
################################
# Extracting Labour Strike data #
################################

import json
import pandas as pd
from pathlib import Path

# Input/output paths
input_path = Path(directory) / "Data" / "Raw" / "Labour Conflict" / "labor_actions_05012026.json"
output_path = Path(directory) / "Data" / "labour_action_tracker_05012026.csv"

# (1) Load JSON directly
with open(input_path, "r", encoding="utf-8") as f:
    data = json.load(f)  # list[dict] or dict

# (2) Normalise into DataFrame
parsed_json_records = pd.DataFrame(data if isinstance(data, list) else [data])

# Swap rows and columns
parsed_json_records  = parsed_json_records .T
# If the column labels are something odd like None, clean that
parsed_json_records .columns.name = None

col_with_dicts = parsed_json_records.columns[0]          # first column
df_flat = pd.json_normalize(parsed_json_records[col_with_dicts])

if "locations" in df_flat.columns:
    loc_df = pd.json_normalize(
        df_flat["locations"].apply(
            lambda x: x[0] if isinstance(x, list) and x else {}
        )
    )
    loc_df = loc_df.add_prefix("Location_")  # e.g. Location_City, Location_State
    df_flat = pd.concat([df_flat.drop(columns=["locations"]), loc_df], axis=1)

# (3) Save to CSV
df_flat.to_csv(output_path, index=False)

print("Data extraction and CSV creation completed successfully.")

clear_env(keep=["directory", "unary_union", "df_flat", "parsed_json_records"])


Data extraction and CSV creation completed successfully.


In [2]:
###################################################
# Cleaning & Manlipuation of Labour Conflict Data #
###################################################

lconflict_event = pd.read_csv(directory + '\\Data\\labour_action_tracker_05012026.csv')

lconflict_event = lconflict_event.replace(r'^\s*$', pd.NA, regex=True) # Replace blank (empty string) values with missing (NaN) in the DataFrame

n_missing = lconflict_event['Employer'].isna().sum()
n_missing_df = lconflict_event[lconflict_event['Employer'].isna()]
print(f"Number of missing observations in Employer: {n_missing}")
lconflict_event = lconflict_event.dropna(subset=['Employer']).reset_index(drop=True) #Dropping rows with missing Employer values

lconflict_event['UniqueID'] = range(len(lconflict_event))

clear_env(keep=["directory", "unary_union", "lconflict_event", "n_missing", "n_missing_df"])

Number of missing observations in Employer: 326


In [3]:
#####################
# (a) - Action Type #
#####################
missing_percent = lconflict_event['Action_type'].isna().mean() * 100
print(f"% of missing data in 'Strike or Protest': {missing_percent:.2f}%")

lconflict_event['Strike'] = (lconflict_event['Action_type'] == 'Strike').astype(int)
lconflict_event['Protest'] = (lconflict_event['Action_type'] == 'Protest').astype(int)
lconflict_event['Labour_unrest'] = ((lconflict_event['Action_type'] == 'Protest') |  (lconflict_event['Action_type'] == 'Strike')).astype(int)

clear_env(keep=["directory", "unary_union", "lconflict_event"])
# Checked if code is correct 08/09/2024

% of missing data in 'Strike or Protest': 0.22%


In [4]:
################
# (B) - Sector #
################
# Step 1: Identify Exact Matches of Strings to NAICS Sectors Descriptions & Create a Dataframe of Unmatched Strings (i.e., multiple sectors or vague descriptions)
import pandas as pd
import re

# read (keep as strings)
NAICS_def = pd.read_excel(
    directory + r"\Data\Raw\Sector\2022-NAICS-Codes-listed-numerically-2-Digit-through-6-Digit.xlsx",
    sheet_name="Two Digit NAICS",
    dtype=str
)

# 2) Normalise column names (strip, remove NBSP, collapse spaces)
NAICS_def.columns = (NAICS_def.columns.astype(str)
                     .str.replace('\xa0', ' ', regex=False)
                     .str.strip()
                     .str.replace(r'\s+', ' ', regex=True))

# 3) Find the title/code columns in a case-insensitive way
cols = {c.lower(): c for c in NAICS_def.columns}
title_col = next(c for k, c in cols.items() if "naics" in k and "title" in k)
code_col  = next(c for k, c in cols.items() if "naics" in k and "code"  in k)

# 4) Adressing issues with slight naming inconsistencies
NAICS_def = NAICS_def.rename(columns={title_col: "Industry", code_col: "NAICS_code"})
NAICS_def = NAICS_def[["Industry", "NAICS_code"]].dropna().reset_index(drop=True)

#------- Naming Corrections -------#
def norm_title(s):
    if pd.isna(s): return ""
    # remove list brackets and quotes: ['manufacturing'] → manufacturing
    s = re.sub(r"[\[\]']", "", s)
    s = str(s).lower().replace("&", " and ")
    s = re.sub(r"[(),]", " ", s)          # drop commas/parentheses
    s = re.sub(r"\s+", " ", s).strip()    # collapse spaces
    s = re.sub(r"t$", "", s)              # remove trailing 't'
    return s

# normalised keys
NAICS_def["key"] = NAICS_def["Industry"].map(norm_title)
lconflict_event["key"] = lconflict_event["Industry"].map(norm_title)

# alias: shortened admin/waste -> official long title
aliases = {
    # already-discussed shortened admin/waste title
    "Administrative and Support and Waste Management":
        "Administrative and Support and Waste Management and Remediation Services",
    # NEW: 'Mining' shorthand to official NAICS sector title
    "Mining":
        "Mining, Quarrying, and Oil and Gas Extraction",
}

alias_rows = []
for short_title, full_title in aliases.items():
    full_key  = norm_title(full_title)
    short_key = norm_title(short_title)
    code_ser  = NAICS_def.loc[NAICS_def["key"].eq(full_key), "NAICS_code"]
    if not code_ser.empty:
        alias_rows.append({
            "Industry":   full_title,
            "NAICS_code": code_ser.iloc[0],
            "key":        short_key
        })

if alias_rows:
    NAICS_def = pd.concat([NAICS_def, pd.DataFrame(alias_rows)], ignore_index=True)

admin_code = NAICS_def.loc[NAICS_def["key"].eq(full_key), "NAICS_code"]
if not admin_code.empty:
    NAICS_def = pd.concat(
        [NAICS_def, pd.DataFrame({"Industry":[full_title],
                                  "NAICS_code":[admin_code.iloc[0]],
                                  "key":[short_key]})],
        ignore_index=True
    )

# keep only distinct key->code for merging
naics_key = NAICS_def[["key", "NAICS_code"]].dropna(subset=["key"]).drop_duplicates("key")

# ------ naming correction ends here ------

# Left-merge: unmatched remain NaN in NAICS_code
lconflict_event = (
    lconflict_event
      .drop(columns=["NAICS_code"], errors="ignore")   # avoid _x/_y
      .merge(naics_key, on="key", how="left")
      .drop(columns=["key"])
)

# Unmatched list
lconflict_event['Multi_sector'] = (
    (lconflict_event['Industry'] != "[]") &
    (lconflict_event['NAICS_code'].isna())
).astype(int)
unmatched_industries = lconflict_event[(lconflict_event['Industry'] != "[]") &
    (lconflict_event['NAICS_code'].isna())]
missing_industries = lconflict_event[lconflict_event['Industry'] == "[]"]

#Dealing with Unmatched Industries (Each string contains multiple sectors)
# Assign Code to each unique sector description, and then weight classification by number of sectors mentioned 
# As a robustness check, we also drop these multi-sector descriptions and re-run the analysis (see Analysis notebook)
# normalised title -> code dictionary (using the 'key' built above)
norm_to_code = (
    NAICS_def[['key', 'NAICS_code']]
    .dropna(subset=['key', 'NAICS_code'])
    .drop_duplicates('key')
    .set_index('key')['NAICS_code']
    .to_dict()
)

def codes_for_multi_cell(text):
    if pd.isna(text):
        return pd.NA
    parts = [p.strip() for p in str(text).split(",") if p.strip()]
    out, seen = [], set()
    for p in parts:
        code = norm_to_code.get(norm_title(p))
        if code and code not in seen:
            out.append(code)
            seen.add(code)
    return ", ".join(out) if out else pd.NA

# New column with all matching NAICS codes, comma-separated
lconflict_event["NAICS_code"] = lconflict_event["NAICS_code"].astype("object")
mask = lconflict_event["Multi_sector"].eq(1)
lconflict_event.loc[mask, "NAICS_code"] = (
    lconflict_event.loc[mask, "Industry"].map(codes_for_multi_cell)
)

lconflict_event['NAICS_code_first'] = lconflict_event['NAICS_code'].str.split(',').str[0]
sectors = lconflict_event['NAICS_code_first'].unique()
print("Unique NAICS codes in the dataset:")
print(sectors)



Unique NAICS codes in the dataset:
['31-33' '42' '71' '51' '61' '62' '72' '23' '54' '92' '44-45' '48-49' '56'
 '21' '22' '53' '11' nan '52' <NA> '55']


In [5]:
# -----------------------------------------------------------------------------
# (3) Agregate to Broad Sectors
# -----------------------------------------------------------------------------
# Map NAICS codes to broad sectors
naics_to_broad = {
    # Goods-producing
    '11': 'Goods-producing',
    '21': 'Goods-producing',
    '22': 'Goods-producing',
    '23': 'Goods-producing',
    '31': 'Goods-producing',
    '32': 'Goods-producing',
    '33': 'Goods-producing',
    # Professional Services
    '51': 'Professional Services',
    '52': 'Professional Services',
    '53': 'Professional Services',
    '54': 'Professional Services',
    '55': 'Professional Services',
    '61': 'Professional Services',
    # Non-Professional Services
    '42': 'Non-Professional Services',
    '44': 'Non-Professional Services',
    '45': 'Non-Professional Services',
    '56': 'Non-Professional Services',
    '71': 'Non-Professional Services',
    '72': 'Non-Professional Services',
    # Transport (separated due to high physical and weather exposure)
    '48': 'Transport',
    '49': 'Transport',
    # Healthcare
    '62': 'Healthcare',
    # Public Sector
    '92': 'Public Sector',
}

# Map NAICS codes to broad sectors
lconflict_event['Broad_Sector'] = (
    lconflict_event['NAICS_code_first']
    .astype(str)
    .str[:2]
    .map(naics_to_broad)
)

In [6]:
# Labour Conflict, Strikes and Protests by Broad Sector
broad_sectors = [
    'Goods-producing', 'Professional Services', 'Non-Professional Services',
    'Transport', 'Healthcare', 'Public Sector'
]

event_types = {
    'Labour_unrest': 'Labour_unrest',
    'Protest': 'Protest',
    'Strike': 'Strike'
}

for sector in broad_sectors:
    sector_clean = sector.replace(' ', '_').replace('-', '_')  # for valid column names
    
    for event_name, event_col in event_types.items():
        col_name = f"{event_name}_{sector_clean}"
        
        lconflict_event[col_name] = (
            (lconflict_event['Broad_Sector'] == sector) &
            (lconflict_event[event_col] == 1)
        ).astype(int)

# Save the cleaned DataFrame for analysis
lconflict_event.to_csv(directory + '\\Data\\lconflict_event.csv', index=False)

In [9]:
###################################################
# Creation of County-Day Panel of Labour Conflict # 
###################################################
# (1) U.S. County Map
US_county_shp = gpd.read_file(directory+'\\Data\\Raw\\US Counties\\US_County_SHP\\c_05mr24.shp')

US_county_shp = (
    US_county_shp.groupby('FIPS')
    .agg({
        'STATE': 'first',
        'CWA': 'first',
        'COUNTYNAME': 'first',
        'TIME_ZONE': 'first',
        'FE_AREA': 'first',
        'LON': 'mean',
        'LAT': 'mean',
        'geometry': lambda x: unary_union(list(x))
    })
    .reset_index()
)
State = US_county_shp[["FIPS", "STATE"]]

US_county_shp = gpd.GeoDataFrame(US_county_shp, geometry='geometry', crs="EPSG:4326")

# (2) U.S. County Map & Location of Labour Unrest Events
event_points = lconflict_event[['Labour_unrest', 'Location_Latitude', 'Location_Longitude']]
#event_points = event_points[~event_points['Location_Longitude'].str.contains(';')] # Removes observations for which we do have multiple lat and long locations
#event_points[['Latitude', 'Longitude']] = event_points['Location_Longitude'].str.split(', ', expand=True)

# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(event_points["Location_Longitude"], event_points["Location_Latitude"])]
event_gdf = gpd.GeoDataFrame(event_points, geometry=geometry, crs="EPSG:4326")

# Reproject to EPSG:5070
event_gdf = event_gdf.to_crs("EPSG:5070")
US_county_shp = US_county_shp.to_crs("EPSG:5070")

# Define geographic bounding boxes for different regions (EPSG:5070) #Use GIS to identify the correct bounding boxes!
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]
alaska_bounds = [-5624079.5091, 3906729.8828, -2123879.3102, 6507096.4794]
hawaii_bounds = [-6338989.6804, 1498912.4257, -5921482.1404, 2141378.3664]
puerto_rico_bounds = [3026333.5643, -81515.6598, 3411431.8432, 76241.1847]

# Separate the regions based on bounding boxes
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]
alaska = US_county_shp.cx[alaska_bounds[0]:alaska_bounds[2], alaska_bounds[1]:alaska_bounds[3]]
hawaii = US_county_shp.cx[hawaii_bounds[0]:hawaii_bounds[2], hawaii_bounds[1]:hawaii_bounds[3]]
puerto_rico = US_county_shp.cx[puerto_rico_bounds[0]:puerto_rico_bounds[2], puerto_rico_bounds[1]:puerto_rico_bounds[3]]

# Separate regions
contiguous_event = event_gdf.cx[contiguous_bounds[0]:contiguous_bounds[2], contiguous_bounds[1]:contiguous_bounds[3]]
alaska_events = event_gdf.cx[alaska_bounds[0]:alaska_bounds[2], alaska_bounds[1]:alaska_bounds[3]]
hawaii_events = event_gdf.cx[hawaii_bounds[0]:hawaii_bounds[2], hawaii_bounds[1]:hawaii_bounds[3]]
puerto_rico_events = event_gdf.cx[puerto_rico_bounds[0]:puerto_rico_bounds[2], puerto_rico_bounds[1]:puerto_rico_bounds[3]]

# (3) Assigning Labour Unrest Events to Counties
events_gdf = gpd.GeoDataFrame(
    lconflict_event,
    geometry=gpd.points_from_xy(lconflict_event["Location_Longitude"], lconflict_event["Location_Latitude"]),
    crs="EPSG:4326"
)

events_gdf = events_gdf.to_crs(epsg=5070)
US_county_shp = US_county_shp.to_crs(events_gdf.crs)

# Matching
matched = gpd.sjoin(events_gdf, US_county_shp, how='left', predicate='within')
unmatched = matched[matched['index_right'].isna()]

# County Panel of Labour Unrest
matched['date'] = pd.to_datetime(matched['Start_date'])
matched['month'] = matched['date'].dt.month
matched['year'] = matched['date'].dt.year

# Broad sectors and event types
broad_sectors = [
    'Goods_producing', 'Professional_Services', 'Non_Professional_Services',
    'Transport', 'Healthcare', 'Public_Sector'
]
event_types = ['Strike', 'Protest', 'Labour_unrest']

# Build aggregation dictionary
agg_dict = {event: 'sum' for event in event_types}
for sector in broad_sectors:
    for event in event_types:
        agg_dict[f"{event}_{sector}"] = 'sum'

unrest_county_D_subset = matched.groupby(['FIPS', 'date']).agg(agg_dict).reset_index()

# Adding County Days with No Labour Unrest
lconflict_event['date'] = pd.to_datetime(lconflict_event['Start_date'])
start_date = lconflict_event['date'].min()
end_date   = lconflict_event['date'].max()
all_days   = pd.date_range(start=start_date, end=end_date, freq='D')

county_df = US_county_shp[["FIPS"]].copy()

all_combinations = pd.MultiIndex.from_product(
    [county_df['FIPS'], all_days], names=['FIPS', 'date']
).to_frame(index=False)

unrest_county_D = pd.merge(
    unrest_county_D_subset, all_combinations, on=['FIPS', 'date'], how='right'
)

# Fill NAs
for event in event_types:
    unrest_county_D[event] = unrest_county_D[event].fillna(0)

for sector in broad_sectors:
    for event in event_types:
        col_name = f"{event}_{sector}"
        unrest_county_D[col_name] = unrest_county_D[col_name].fillna(0)

# 'At least one event' indicator
event_keys = unrest_county_D_subset[['FIPS']].drop_duplicates()
event_keys['at least one event'] = 1

unrest_county_D = unrest_county_D.merge(event_keys, on=['FIPS'], how='left')
unrest_county_D['at least one event'] = unrest_county_D['at least one event'].fillna(0).astype(int)

unrest_county_D = unrest_county_D.merge(State, on=['FIPS'], how='left')

# Save
unrest_county_D.to_csv(directory + '\\Data\\labour_unrest_county_day_panel.csv', index=False)

In [21]:
####################################
######## Creating Event-Day Panel ##
####################################
lconflict_event_filtered = lconflict_event[['id', 'Employer', 'Action_type', 'Strike', 'Protest', 'NAICS_code', 'Broad_Sector', 
                                            'Start_date', 'End_date', 'Worker_demands', 'Duration', 'Location_Latitude', 
                                            'Location_Longitude', 'Approximate_Number_of_Participants', 
                           'Bargaining_Unit_Size', 'Labour_unrest_Goods_producing', 'Labour_unrest_Professional_Services', 
                           'Labour_unrest_Non_Professional_Services', 'Labour_unrest_Transport', 'Labour_unrest_Healthcare', 
                           'Labour_unrest_Public_Sector']].copy()
lconflict_event_filtered['Duration'] = lconflict_event_filtered['Duration'].abs()
lconflict_event_filtered['Start_date'] = pd.to_datetime(lconflict_event_filtered['Start_date'], errors='coerce')
lconflict_event_filtered['End_date'] = pd.to_datetime(lconflict_event_filtered['End_date'], errors='coerce')

#!# Duration of Strikes & Num. of Demands #!#
# Duration 
# Summary statistics for Duration by action type
summary = (
    lconflict_event_filtered
    .groupby("Action_type")["Duration"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Std="std",
        Min="min",
        Max="max",
        Q25=lambda x: x.quantile(0.25),
        Q75=lambda x: x.quantile(0.75),
    )
    .round(2)
    .reset_index()
)

print(summary.to_string(index=False))
    
# Worker demands
lconflict_event_filtered['Num_Demands'] = lconflict_event_filtered['Worker_demands'].apply(lambda x: len(str(x).split(',')) if pd.notna(x) else 0)
# Summary statistics for Num_Demands by action type
demand_summary = (
    lconflict_event_filtered
    .groupby("Action_type")["Num_Demands"]
    .agg(
        Count="count",
        Mean="mean",
        Median="median",
        Std="std",
        Min="min",
        Max="max",
        Q25=lambda x: x.quantile(0.25),
        Q75=lambda x: x.quantile(0.75),
    )
    .round(2)
    .reset_index()
)
print(demand_summary.to_string(index=False))

##################################################
# Creation of Event-Day Panel of Labour Conflict # 
###################################################
# (1) U.S. County Map
US_county_shp = gpd.read_file(directory+'\\Data\\Raw\\US Counties\\US_County_SHP\\c_05mr24.shp')

US_county_shp = (
    US_county_shp.groupby('FIPS')
    .agg({
        'STATE': 'first',
        'CWA': 'first',
        'COUNTYNAME': 'first',
        'TIME_ZONE': 'first',
        'FE_AREA': 'first',
        'LON': 'mean',
        'LAT': 'mean',
        'geometry': lambda x: unary_union(list(x))
    })
    .reset_index()
)
State = US_county_shp[["FIPS", "STATE"]]

US_county_shp = gpd.GeoDataFrame(US_county_shp, geometry='geometry', crs="EPSG:4326")

# (2) U.S. County Map & Location of Labour Unrest Events
event_points = lconflict_event_filtered[['Location_Latitude', 'Location_Longitude']]
#event_points = event_points[~event_points['Location_Longitude'].str.contains(';')] # Removes observations for which we do have multiple lat and long locations
#event_points[['Latitude', 'Longitude']] = event_points['Location_Longitude'].str.split(', ', expand=True)

# Convert to GeoDataFrame
geometry = [Point(xy) for xy in zip(event_points["Location_Longitude"], event_points["Location_Latitude"])]
event_gdf = gpd.GeoDataFrame(event_points, geometry=geometry, crs="EPSG:4326")

# Reproject to EPSG:5070
event_gdf = event_gdf.to_crs("EPSG:5070")
US_county_shp = US_county_shp.to_crs("EPSG:5070")

# Define geographic bounding boxes for different regions (EPSG:5070) #Use GIS to identify the correct bounding boxes!
contiguous_bounds = [-2453691.2025, 171085.9341, 2365756.5659, 3381509.2103]

# Separate the regions based on bounding boxes
contiguous = US_county_shp.cx[contiguous_bounds[0]:contiguous_bounds[2],contiguous_bounds[1]:contiguous_bounds[3]]

# Separate regions
contiguous_event = event_gdf.cx[contiguous_bounds[0]:contiguous_bounds[2], contiguous_bounds[1]:contiguous_bounds[3]]

# (3) Assigning Labour Unrest Events to Counties
events_gdf = gpd.GeoDataFrame(
    lconflict_event_filtered,
    geometry=gpd.points_from_xy(lconflict_event_filtered["Location_Longitude"], lconflict_event_filtered["Location_Latitude"]),
    crs="EPSG:4326"
)

events_gdf = events_gdf.to_crs(epsg=5070)
US_county_shp = US_county_shp.to_crs(events_gdf.crs)

# Matching
matched = gpd.sjoin(events_gdf, US_county_shp, how='left', predicate='within')
unmatched = matched[matched['index_right'].isna()]

# County Panel of Labour Unrest
matched['date'] = pd.to_datetime(matched['Start_date'])
matched['month'] = matched['date'].dt.month
matched['year'] = matched['date'].dt.year

event_day_panel = matched[['id', 'Employer', 'date', 'year', 'month', 'Duration', 'Num_Demands', 'Approximate_Number_of_Participants', 
                           'Bargaining_Unit_Size', 'Strike', 'Protest', 'Labour_unrest_Goods_producing', 'Labour_unrest_Professional_Services', 
                           'Labour_unrest_Non_Professional_Services', 'Labour_unrest_Transport', 'Labour_unrest_Healthcare', 
                           'Labour_unrest_Public_Sector', 'FIPS', 'STATE']].copy()

# Save
event_day_panel.to_csv(directory + '\\Data\\labour_unrest_event_day_panel.csv', index=False)

Action_type  Count  Mean  Median   Std  Min    Max  Q25  Q75
    Protest   2326  1.30     1.0  3.05  1.0   93.0  1.0  1.0
     Strike   1634 14.82     2.5 48.29  1.0 1133.0  1.0  9.0
Action_type  Count  Mean  Median  Std  Min  Max  Q25  Q75
    Protest   2350  2.11     2.0 1.08    1    7  1.0  3.0
     Strike   1751  2.17     2.0 1.10    1    7  1.0  3.0


In [ ]:
print(matched)